In [ ]:
import os
import json
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from utils.arxiv_search import build_arxiv_search_url, run_scraper_in_background, scrape_arxiv_url_to_json
from utils.trafilatura_processor import process_arxiv_url_with_trafilatura

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)

arxiv_config = {
    "topics": {
        "AI": "Artificial Intelligence",
        "ML": "Machine Learning",
        "NN": "Neural Networks",
        "CV": "Computer Vision",
        "NLP": "Natural Language Processing"
    },
    "page_sizes": [200],
    "file_locations": {
        "raw": "src/raw/",
        "clean": "src/clean/"
    }
}

# --- Step 1: Scrape the initial search results ---
for key, query in arxiv_config["topics"].items():
    for size in arxiv_config["page_sizes"]:
        url = build_arxiv_search_url(query=query, size=size)
        raw_path = os.path.join(arxiv_config['file_locations']['raw'], f"arxiv_{key.lower()}_{size}.json")
        logging.info(f"Scraping search results for '{query}' to {raw_path}")
        scrape_arxiv_url_to_json(url, output_file=raw_path)

# --- Step 2: Process each abstract page with Trafilatura ---
all_cleaned_papers = []
for key in arxiv_config["topics"]:
    for size in arxiv_config["page_sizes"]:
        raw_path = os.path.join(arxiv_config['file_locations']['raw'], f"arxiv_{key.lower()}_{size}.json")
        if not os.path.exists(raw_path):
            logging.warning(f"File not found: {raw_path}. Skipping.")
            continue

        with open(raw_path, 'r', encoding='utf-8') as f:
            papers = json.load(f)
        
        urls = [paper.get("link") for paper in papers if paper.get("link")]
        
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_url = {executor.submit(process_arxiv_url_with_trafilatura, url): url for url in urls}
            for future in as_completed(future_to_url):
                result = future.result()
                if result:
                    all_cleaned_papers.append(result)

# --- Step 3: Save the final cleaned data ---
final_output_file = 'arxiv_clean.json'
with open(final_output_file, 'w', encoding='utf-8') as f:
    json.dump(all_cleaned_papers, f, indent=2, ensure_ascii=False)

logging.info(f"Saved {len(all_cleaned_papers)} cleaned papers to {final_output_file}")

2025-11-04 12:21:12,485 [DEBUG] minimum date setting: 1995-01-01 00:00:00
2025-11-04 12:21:12,494 [INFO] Scraping search results for 'Artificial Intelligence' to src/raw/arxiv_ai_200.json
2025-11-04 12:21:18,295 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-04 12:21:18,763 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=artificial+intelligence&searchtype=all&abstracts=show&order=-announced_date_first&size=200 HTTP/1.1" 200 940079
2025-11-04 12:21:19,014 [DEBUG] Saved 200 papers to src/raw/arxiv_ai_200.json
2025-11-04 12:21:19,015 [DEBUG] Type of output file:<class 'str'>
2025-11-04 12:21:19,016 [INFO] Scraping search results for 'Machine Learning' to src/raw/arxiv_ml_200.json
2025-11-04 12:21:24,196 [DEBUG] Starting new HTTPS connection (1): arxiv.org:443
2025-11-04 12:21:25,449 [DEBUG] https://arxiv.org:443 "GET /search/cs?query=machine+learning&searchtype=all&abstracts=show&order=-announced_date_first&size=200 HTTP/1.1" 200 921879
2025-11-04 12:21:25,753 [DEBUG